In [ ]:
import os
import os.path as osp
import warnings
from math import pi as PI
from typing import Optional
import ase
import numpy as np
import torch
import torch.nn.functional as F
from torch.nn import Embedding, Linear, ModuleList, Sequential
from torch_scatter import scatter
from torch_geometric.nn import radius_graph

class SchNet(torch.nn.Module):
    r"""The continuous-filter convolutional neural network SchNet from the
    `"SchNet: A Continuous-filter Convolutional Neural Network for Modeling
    Quantum Interactions" <https://arxiv.org/abs/1706.08566>`_ paper that uses
    the interactions blocks of the form

    .. math::
        \mathbf{x}^{\prime}_i = \sum_{j \in \mathcal{N}(i)} \mathbf{x}_j \odot
        h_{\mathbf{\Theta}} ( \exp(-\gamma(\mathbf{e}_{j,i} - \mathbf{\mu}))),

    here :math:`h_{\mathbf{\Theta}}` denotes an MLP and
    :math:`\mathbf{e}_{j,i}` denotes the interatomic distances between atoms.

    Args:
        hidden_channels (int, optional): Hidden embedding size.
            (default: :obj:`128`)
        num_filters (int, optional): The number of filters to use.
            (default: :obj:`128`)
        num_interactions (int, optional): The number of interaction blocks.
            (default: :obj:`6`)
        num_gaussians (int, optional): The number of gaussians :math:`\mu`.
            (default: :obj:`50`)
        cutoff (float, optional): Cutoff distance for interatomic interactions.
            (default: :obj:`10.0`)
        max_num_neighbors (int, optional): The maximum number of neighbors to
            collect for each node within the :attr:`cutoff` distance.
            (default: :obj:`32`)
        readout (string, optional): Whether to apply :obj:`"add"` or
            :obj:`"mean"` global aggregation. (default: :obj:`"add"`)
        dipole (bool, optional): If set to :obj:`True`, will use the magnitude
            of the dipole moment to make the final prediction, *e.g.*, for
            target 0 of :class:`torch_geometric.datasets.QM9`.
            (default: :obj:`False`)
        mean (float, optional): The mean of the property to predict.
            (default: :obj:`None`)
        std (float, optional): The standard deviation of the property to
            predict. (default: :obj:`None`)
        atomref (torch.Tensor, optional): The reference of single-atom
            properties.
            Expects a vector of shape :obj:`(max_atomic_number, )`.
    """

    def __init__(self, hidden_channels: int = 128, num_filters: int = 128,
                 num_interactions: int = 6, num_gaussians: int = 50,
                 cutoff: float = 5.0, max_num_neighbors: int = 32,
                 readout: str = 'add', dipole: bool = False,
                 mean: Optional[float] = None, std: Optional[float] = None,
                 atomref: Optional[torch.Tensor] = None):
        super().__init__()

        

        self.hidden_channels = hidden_channels
        self.num_filters = num_filters
        self.num_interactions = num_interactions
        self.num_gaussians = num_gaussians
        self.cutoff = cutoff
        self.max_num_neighbors = max_num_neighbors
        self.readout = readout
        self.dipole = dipole
        self.readout = 'add' if self.dipole else self.readout
        self.mean = mean
        self.std = std
        self.scale = None

        atomic_mass = torch.from_numpy(ase.data.atomic_masses)

        self.embedding = Embedding(100, hidden_channels)
        self.distance_expansion = GaussianSmearing(0.0, cutoff, num_gaussians)

        self.interactions = ModuleList()
        for _ in range(num_interactions):
            block = InteractionBlock(hidden_channels, num_gaussians,
                                     num_filters, cutoff)
            self.interactions.append(block)

        self.lin1 = Linear(hidden_channels, hidden_channels // 2)
        self.act = ShiftedSoftplus()
        self.lin2 = Linear(hidden_channels // 2, 1)
        self.reset_parameters()


    def reset_parameters(self):
        self.embedding.reset_parameters()
        for interaction in self.interactions:
            interaction.reset_parameters()
        torch.nn.init.xavier_uniform_(self.lin1.weight)
        self.lin1.bias.data.fill_(0)
        torch.nn.init.xavier_uniform_(self.lin2.weight)
        self.lin2.bias.data.fill_(0)

    def forward(self, z, pos, batch=None):
        """"""
        assert z.dim() == 1 and z.dtype == torch.long
        batch = torch.zeros_like(z) if batch is None else batch
        pos.requires_grad = True

        h = self.embedding(z) #TODO: compute initial node representations
        edge_index = radius_graph(pos, r=self.cutoff, batch=batch, max_num_neighbors=self.max_num_neighbors)
        
        edge_weight = torch.norm(pos[edge_index[1]]-pos[edge_index[0]], dim= -1) #TODO: compute pairwise distances
        edge_attr = self.distance_expansion(edge_weight) #TODO: compute radial basis expansion of pairwise distances

        for interaction in self.interactions:
            #TODO: call interaction layers
            h = interaction.forward(h, edge_index, edge_weight, edge_attr)

        #linear projection to obtain atomwise energies
        h = self.lin1(h)
        h = self.act(h)
        h = self.lin2(h)
        energy_v = scatter(h, batch, dim=0, reduce="sum") #TODO: compute global energies per molecule
        print(energy_v.requires_grad)
        energy = torch.sum(energy_v)

        #forces = torch.autograd.grad(torch.sum(energy_v), pos)#TODO: compute forces
        forces = torch.autograd.grad(torch.sum(energy_v), pos, create_graph=True)[0]

        return energy, forces


class InteractionBlock(torch.nn.Module):
    def __init__(self, hidden_channels, num_gaussians, num_filters, cutoff):
        super().__init__()
        self.mlp = Sequential(
            Linear(num_gaussians, num_filters),
            ShiftedSoftplus(),
            Linear(num_filters, num_filters),
        )
        self.conv = CFConv(hidden_channels, hidden_channels, num_filters,
                           self.mlp, cutoff)
        self.act = ShiftedSoftplus()
        self.lin = Linear(hidden_channels, hidden_channels)

        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.xavier_uniform_(self.mlp[0].weight)
        self.mlp[0].bias.data.fill_(0)
        torch.nn.init.xavier_uniform_(self.mlp[2].weight)
        self.mlp[2].bias.data.fill_(0)
        self.conv.reset_parameters()
        torch.nn.init.xavier_uniform_(self.lin.weight)
        self.lin.bias.data.fill_(0)

    def forward(self, h, edge_index, edge_weight, edge_attr):
        h = self.conv(h, edge_index, edge_weight, edge_attr) #TODO: fill in this line
        h = self.act(h)
        h = self.lin(h)
        return h


class CFConv(torch.nn.Module):
    def __init__(self, in_channels, out_channels, num_filters, nn, cutoff):
        super().__init__()
        self.lin1 = Linear(in_channels, num_filters, bias=False)
        self.lin2 = Linear(num_filters, out_channels)
        self.nn = nn
        self.cutoff = cutoff
        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.xavier_uniform_(self.lin1.weight)
        torch.nn.init.xavier_uniform_(self.lin2.weight)
        self.lin2.bias.data.fill_(0)

    def forward(self, h, edge_index, edge_weight, edge_attr):
        h = self.lin1(h)
        c = 0.5 * (torch.cos( (edge_weight) / self.cutoff * torch.pi) + 1) #TODO: transform edge weights to [0,1]
        W = self.nn(edge_attr)
        message = c.unsqueeze(-1) * (W * h[edge_index[0]])
        #print(message)
        #print(h)
        #print(edge_index[1])
        #print(h[edge_index[1]].to(torch.int64))
        agg = scatter(message, edge_index[1], dim=0, reduce="sum")
        h =  h + agg #TODO: perform graph convolution
        h = self.lin2(h)
        return h
        

class GaussianSmearing(torch.nn.Module):
    def __init__(self, start=0.0, stop=5.0, num_gaussians=50):
        super().__init__()
        offset = torch.linspace(start, stop, num_gaussians)
        self.gamma = 0.5 / (offset[1] - offset[0]).item()**2
        self.register_buffer('offset', offset) #sets the attribute self.offset = offset

    def forward(self, dist):
        #TODO: implement this function
        RBFk = torch.exp(-self.gamma * ((dist.unsqueeze(-1)) - self.offset) ** 2)
        return RBFk
    
class ShiftedSoftplus(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.shift = torch.log(torch.tensor(2.0)).item()

    def forward(self, x):
        return F.softplus(x) - self.shift


In [ ]:
import torch
import numpy as np
import argparse
from schnet import SchNet
from lmdb_dataset import LmdbDataset, data_list_collater
from torch.utils.data import DataLoader
from tqdm import tqdm
from utils import L2MAELoss, rotate_3d_coordinates
import wandb
def train(data_dir, size, r_max, batch_size, lr, max_epochs, rotated, device):

    #W&B Run
    wandb.login()
    run = wandb.init(
    # Set the project where this run will be logged
    project="dl-physics-hw2-schnet",
    name=f"aspirin_{size}_r_max={r_max}_rot={rotated}",
    # Track hyperparameters and run metadata
    config={
        "learning_rate": lr,
        "epochs": max_epochs,
        "batch_size": batch_size,
        "r_max": r_max,
        "size": size
    }
    )

    # data
    train_dir = f"{data_dir}/{size}/train"
    val_dir = f"{data_dir}/{size}/val"
    test_dir = f"{data_dir}/{size}/test"
    train_dataset = LmdbDataset({'src': train_dir})
    val_dataset = LmdbDataset({'src': val_dir})
    test_dataset = LmdbDataset({'src': test_dir})
    train_dataloader = DataLoader(train_dataset, collate_fn=data_list_collater, \
                                batch_size = batch_size, shuffle = False)
    val_dataloader = DataLoader(val_dataset, collate_fn=data_list_collater, \
                                batch_size = batch_size, shuffle = False)
    test_dataloader = DataLoader(test_dataset, collate_fn=data_list_collater, \
                                batch_size = batch_size, shuffle = False)
    # model
    model = SchNet(cutoff = r_max)
    model = model.to(device)

    #optimization
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.2, patience=10)    
    force_loss = L2MAELoss()

    epoch = 0
    while epoch < max_epochs and optimizer.param_groups[0]['lr'] > 1e-7:
        model.train()
        print(f"Train Epoch {epoch + 1}")
        for batch in tqdm(train_dataloader):
            #TODO: fill in the training loop
            
            
            atomic_numbers = batch['atomic_numbers']
            positions = batch['pos']
            if rotated:
                positions = rotate_3d_coordinates(positions,30,30,30)
            
            indices = batch['batch']
            data_forces = batch['force']
            optimizer.zero_grad()
            predictedE, predictedF = model(atomic_numbers, positions, indices)
            data_forces_rot = rotate_3d_coordinates(data_forces,30,30,30)
            loss = force_loss(predictedF[0], data_forces_rot)
            loss.backward()
            optimizer.step()
            wandb.log({"train_force_loss": loss, "lr": optimizer.param_groups[0]['lr']})
            wandb.log({"train_energy": predictedE})
        loss = 0
        val_loss = 0
        model.eval()
        print(f"Val Epoch {epoch + 1}")
        for batch in val_dataloader:
            #TODO: fill in validation loop
            atomic_numbers = batch['atomic_numbers']
            positions = batch['pos']
            positions = rotate_3d_coordinates(positions,30,30,30)
            indices = batch['batch']
            data_forces = batch['force']
            energy_pred, forces_pred = model(atomic_numbers, positions, indices)
            data_forces_rot = rotate_3d_coordinates(data_forces,30,30,30)

            loss = force_loss(forces_pred, data_forces_rot)
            val_loss += loss
            wandb.log({"val_force_loss": loss})
            wandb.log({"val_energy": energy_pred})
        mean_val_loss = val_loss / len(val_dataloader)
        scheduler.step(mean_val_loss)
        
        epoch +=1
    
    #Final Test
    test_losses = []
    for batch in test_dataloader:
        #TODO: fill in test loop
        test_losses.append(loss.mean().item())
    wandb.log({"test_force_loss": sum(test_losses) / len(test_losses)})

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('--seed', type=int, default=123, help='Random seed')
    parser.add_argument('--size', type=str, default='1k', help='Size of dataset')
    parser.add_argument('--r_max', type=float, default=5.0, help='Cutoff (A) for radius graph construction')
    parser.add_argument('--lr', type=float, default=0.001, help='Learning rate')
    parser.add_argument('--data_dir', type=str, default="/Users/alankstoev/Documents/VS Code/CS294/homework2/homework2/problem4-schnet 2/aspirin", help='Directory of data')
    parser.add_argument('--batch_size', type=int, default=128, help='Batch size')
    parser.add_argument('--max_epochs', type=int, default=1000, help='Number of epochs')
    parser.add_argument('--rot', type=bool, default=False, help='Rotated Positions')

    args = parser.parse_args()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    train(args.data_dir, args.size, args.r_max, args.batch_size, args.lr, args.max_epochs, args.rot, device)


In [ ]:
''' #4 WORD RESPONSES 
i. This is a continuous filter convolution because it uses a continuous function (exponentials)
for the kernel (the MLP applied to the edge attributes). The GaussianSmearing function is used 
for this specific purpose (along with decorrelating the distnaces to a higher degree according to
the paper)

ii. The cutoff radius used in radius_graph is the hyperparameter that dicatates how far information 
is propogated to the central node for each local neighborhood over which message passing occurs. r_cutoff
dictates this by explicitly creating graphs based on this raidus while defining the graph for each atom
which is then used to calculate edge information and attributes to be used in the CFConv class. The
number of interaction blocks appended to the interactions modules list also plays a role as it dictates
the number of convolutions applied to each local neighborhood.

iii. see code above

iv. see graphs attached, the force losses and energies are approximately the same, however, not exactly:
Force Loss  r_max = 3.0 rot=True: 1.8170
Force Loss  r_max = 3.0 rot=False: 1.8171
Energy Predicted r_max = 3.0 rot=True: 13
Energy Predicted r_max = 3.0 rot=False: 12.7

v. see graphs attached

vi. 
Force Loss  aspirin_1k_r_max=3.0_rot=False	1.8178627490997314
Force Loss  aspirin_1k_r_max=4.0	1.8172802925109863
Force Loss  aspirin_1k_r_max=5.0	1.816683053970337
Force Loss  asspirin_1k_r_max=6.0	1.8158748149871826

Increasing r_max generally increases training time and seems to have a general trend of better accuracy. 





'''